In [2]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

In [3]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [4]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [5]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [6]:
# Check null values in D1
clinical_train.isnull().sum().sum()

0

In [7]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

### COX assumption in Train data

In [8]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.00001)
cph.fit(df, duration_col='OS', event_col='event_OS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>
         test_name = proportional_hazard_test

---
                test_statistic      p  -log2(p)
MTV                       0.07   0.80      0.33
SUVpeak                   0.17   0.68      0.56
TLG                       0.38   0.54      0.90
age                       0.09   0.77      0.38
cavum_oris                0.00   0.99      0.02
charlson                  1.66   0.20      2.34
female                    2.67   0.10      3.29
histgrade_high            0.13   0.72      0.48
hpv_related              12.67 <0.005     11.39
hypopharynx               0.00   0.99      0.01
larynx                    0.00   0.98      0.03
oropharynx                0.00   0.98      0.02
pack_years                0.06   0.81      0.31
uicc8_III-IV              

In [9]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['hpv_related'], dtype='object')


In [10]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.1)
cph.fit(df, duration_col='OS', event_col='event_OS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>
         test_name = proportional_hazard_test

---
                test_statistic    p  -log2(p)
MTV                       0.59 0.44      1.17
SUVpeak                   0.01 0.91      0.14
TLG                       0.00 0.96      0.06
age                       0.43 0.51      0.97
cavum_oris                0.41 0.52      0.94
charlson                  0.87 0.35      1.51
female                    2.39 0.12      3.04
histgrade_high            0.11 0.74      0.44
hpv_related               6.46 0.01      6.51
hypopharynx               0.04 0.84      0.25
larynx                    0.97 0.33      1.62
oropharynx                0.85 0.36      1.49
pack_years                0.00 0.99      0.01
uicc8_III-IV              0.00 0.95      0.07


In [11]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['hpv_related'], dtype='object')


###### penalizer values essentially result in same result 

## Test dataset: MAASTRO 

In [12]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [13]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [14]:
# need to choose patient_id from MAASTRO_D1 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [15]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,OS,OS_event
0,1,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342,44.43,1.0
3,4,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979,37.20,1.0
4,6,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782,19.00,1.0
95,111,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492,58.93,0.0


In [16]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [17]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,OS,OS_event


In [18]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [19]:
# Set lower, upper time point and gbsg_times for IBS calculation later 
lower, upper = np.percentile(y['OS'], [10, 90])
gbsg_times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [20]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 14)
y_train:  (139,)


In [21]:
clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

In [22]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 16)

In [23]:
# VIF dataframe 
vif_data = pd.DataFrame() 
vif_data["feature"] = X.columns 
  
# calculating VIF for each feature 
vif_data["VIF"] = [variance_inflation_factor(X.values, i) 
                          for i in range(len(X.columns))] 

vif_data

,feature,VIF
0,age,1.136852
1,female,1.197250
2,cavum_oris,7.993011
3,oropharynx,60.199385
4,hypopharynx,11.118710
5,larynx,14.183474
6,histgrade_high,1.150479
7,hpv_related,4.309385
8,charlson,1.302350
9,pack_years,1.647441


# Standardization

In [24]:
original_X = X.copy()

In [25]:
# Standardize the data 
## Save the column and index 
X_columns = X.columns 
X_index = X.index

## Standardize the data but not the categorical columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

### Separate the categorical and non-categorical columns
X_categorical = X[categorical_columns]
X_numeric = X.drop(categorical_columns, axis=1)
X_numeric_columns = X_numeric.columns
X_numeric_index = X_numeric.index

### Standardize non-categorical and then concat with the categorical
scaler = MinMaxScaler()  
X_numeric_std = scaler.fit_transform(X_numeric)
X_numeric_std = pd.DataFrame(X_numeric_std, columns=X_numeric_columns, index=X_numeric_index)
X_std = pd.concat([X_categorical, X_numeric_std], axis=1)

## Sort the order of the columns as it was in the clinical train 
X_std = X_std[original_X.columns]

In [26]:
# Standardize X_MAASTRO 
MAASTRO_new = X_MAASTRO.copy()
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [27]:
# Saving the data 
X_new = X 
X_new_std = X_std 
MAASTRO_new = X_MAASTRO 
MAASTRO_new_std = MAASTRO_new_std 

In [28]:
X_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,54.238356,1,0,1,0,0,1,0.0,0,0.000000,0.0,14.473272,7.934,86.228420
1,54.539726,0,0,0,0,1,0,0.0,1,27.404795,0.0,5.044678,1.656,7.040100
2,59.019178,0,1,0,0,0,1,0.0,1,41.019178,1.0,7.839043,14.502,83.569669
3,70.726027,0,0,0,0,1,0,0.0,1,37.500000,0.0,2.880631,2.440,5.567091
4,67.865753,0,0,0,0,1,0,0.0,1,53.000000,0.0,5.402006,3.668,16.150550
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,60.435616,0,0,1,0,0,1,1.0,0,0.000000,0.0,9.290139,3.650,26.280140
135,68.794521,0,0,1,0,0,1,1.0,0,0.000000,1.0,7.172883,18.967,101.754834
136,57.498630,0,0,1,0,0,1,1.0,1,39.498630,0.0,13.873187,6.370,66.273201
137,65.684932,0,0,1,0,0,1,1.0,1,71.527397,1.0,7.507419,12.443,71.832443


In [29]:
X_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,0.365347,1,0,1,0,0,1,0.0,0,0.000000,0.0,0.435155,0.074159,0.044798
1,0.373041,0,0,0,0,1,0,0.0,1,0.213981,0.0,0.089331,0.008512,0.001900
2,0.487409,0,1,0,0,0,1,0.0,1,0.320284,1.0,0.191823,0.142839,0.043358
3,0.786304,0,0,0,0,1,0,0.0,1,0.292806,0.0,0.009957,0.016710,0.001102
4,0.713276,0,0,0,0,1,0,0.0,1,0.413832,0.0,0.102437,0.029551,0.006835
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,0.523573,0,0,1,0,0,1,1.0,0,0.000000,0.0,0.245047,0.029363,0.012323
135,0.736989,0,0,1,0,0,1,1.0,0,0.000000,1.0,0.167389,0.189529,0.053209
136,0.448587,0,0,1,0,0,1,1.0,1,0.308411,0.0,0.413145,0.057805,0.033988
137,0.657597,0,0,1,0,0,1,1.0,1,0.558497,1.0,0.179660,0.121309,0.037000


In [30]:
MAASTRO_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623
1,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700
2,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342
3,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979
4,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782
95,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868
96,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274
97,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492


In [31]:
MAASTRO_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,0.384793,0,0,1,0,0,1,1,1,0.000000,0,0.470560,0.230038,0.140892
1,0.384793,0,0,1,0,0,0,0,0,0.156163,1,0.228146,0.050381,0.018119
2,0.384793,0,0,1,0,0,0,0,1,0.046849,1,0.398581,0.072664,0.038519
3,0.537983,1,0,0,0,1,1,0,1,0.351367,1,0.220934,0.073887,0.023434
4,0.767767,0,0,1,0,0,1,1,1,0.460681,0,0.263159,0.150525,0.056396
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,0.665641,1,0,0,0,1,0,0,0,0.429449,1,1.053738,0.055086,0.076361
95,0.589046,0,0,0,0,1,0,0,1,1.358619,1,0.382644,0.066296,0.035582
96,0.589046,0,0,1,0,0,1,1,1,0.000000,1,0.232370,0.163554,0.053664
97,0.359261,0,0,1,0,0,1,1,0,0.000000,0,0.424553,0.095564,0.054008


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [32]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                gbsg_times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(gbsg_times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, gbsg_times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-20 17:05:32,957] A new study created in memory with name: no-name-f25ecdb0-1408-4eb6-8725-ca3ef08c1083


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-20 17:05:33,345] A new study created in memory with name: no-name-257fa157-38e9-4686-9042-968a6e35b79e


Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8523206751054853
Fold 5 C-index: 0.6525821596244131
[I 2024-04-20 17:05:33,340] Trial 0 finished with value: 0.7580581707147598 and parameters: {}. Best is trial 0 with value: 0.7580581707147598.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7580581707147598], datetime_start=datetime.datetime(2024, 4, 20, 17, 5, 33, 73745), datetime_complete=datetime.datetime(2024, 4, 20, 17, 5, 33, 340311), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7580581707147598


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.18224790157453521
Fold 2 IBS: 0.1793618517283866
Fold 3 IBS: 0.5169743692261356
Fold 4 IBS: 0.6484315948601662
Fold 5 IBS: 0.2737533899229805
[I 2024-04-20 17:05:33,755] Trial 0 finished with value: 0.3601538214624408 and parameters: {}. Best is trial 0 with value: 0.3601538214624408.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.3601538214624408], datetime_start=datetime.datetime(2024, 4, 20, 17, 5, 33, 395749), datetime_complete=datetime.datetime(2024, 4, 20, 17, 5, 33, 755532), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.3601538214624408


In [33]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [34]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.758
train_ibs:  0.36


#### Test

In [35]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [36]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

ValueError: search direction contains NaN or infinite values

In [37]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [38]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

NameError: name 'c_index' is not defined

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [39]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                gbsg_times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(gbsg_times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, gbsg_times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-20 17:05:44,958] A new study created in memory with name: no-name-2e5b6053-4ef8-4a5f-b432-aa77d5ec5d5b


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-20 17:05:45,083] A new study created in memory with name: no-name-63b23f58-4d44-43a4-bd3e-fab293e59feb


Fold 1 C-index: 0.7424242424242424
Fold 2 C-index: 0.6383928571428571
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.8016877637130801
Fold 5 C-index: 0.6197183098591549
[I 2024-04-20 17:05:45,079] Trial 0 finished with value: 0.7182877718827688 and parameters: {}. Best is trial 0 with value: 0.7182877718827688.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7182877718827688], datetime_start=datetime.datetime(2024, 4, 20, 17, 5, 44, 994942), datetime_complete=datetime.datetime(2024, 4, 20, 17, 5, 45, 79146), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7182877718827688


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397651763328035
Fold 2 IBS: 0.22157791469499555
Fold 3 IBS: 0.20453594133219627
Fold 4 IBS: 0.22473803324306438
Fold 5 IBS: 0.21812431436306018
[I 2024-04-20 17:05:45,246] Trial 0 finished with value: 0.21659054425331936 and parameters: {}. Best is trial 0 with value: 0.21659054425331936.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21659054425331936], datetime_start=datetime.datetime(2024, 4, 20, 17, 5, 45, 115219), datetime_complete=datetime.datetime(2024, 4, 20, 17, 5, 45, 246564), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21659054425331936


In [40]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [41]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.718
train_ibs:  0.217


#### Test

In [42]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [43]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
gbsg_times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(gbsg_times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, gbsg_times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.624


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [44]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [107]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                gbsg_times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(gbsg_times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, gbsg_times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-21 11:10:00,529] A new study created in memory with name: no-name-b1a1ab23-5ca1-4902-952c-104b3bd7e35a


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8481012658227848


[I 2024-04-21 11:10:00,854] A new study created in memory with name: no-name-f6625b63-12ea-4961-b7b3-7ce2a81e100e


Fold 5 C-index: 0.6525821596244131
[I 2024-04-21 11:10:00,849] Trial 0 finished with value: 0.7563214317153626 and parameters: {}. Best is trial 0 with value: 0.7563214317153626.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7563214317153626], datetime_start=datetime.datetime(2024, 4, 21, 11, 10, 0, 553406), datetime_complete=datetime.datetime(2024, 4, 21, 11, 10, 0, 848722), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7563214317153626


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.1822562210458609
Fold 2 IBS: 0.17929926098802096
Fold 3 IBS: 0.15678124122148837
Fold 4 IBS: 0.14669965181722588
Fold 5 IBS: 0.272560860180458
[I 2024-04-21 11:10:01,234] Trial 0 finished with value: 0.18751944705061083 and parameters: {}. Best is trial 0 with value: 0.18751944705061083.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.18751944705061083], datetime_start=datetime.datetime(2024, 4, 21, 11, 10, 0, 886321), datetime_complete=datetime.datetime(2024, 4, 21, 11, 10, 1, 234427), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.18751944705061083


In [108]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [109]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.756
train_ibs:  0.188


#### Test

In [110]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [111]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
gbsg_times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(gbsg_times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, gbsg_times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.583


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.287


In [112]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [51]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                gbsg_times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(gbsg_times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, gbsg_times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-20 17:05:49,495] A new study created in memory with name: no-name-0e51ccfa-8249-4b48-a610-1f979205fc9e


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6525821596244131
[I 2024-04-20 17:05:49,874] Trial 0 finished with value: 0.7563214317153626 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7563214317153626.
Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7366071428571429
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.6525821596244131
[I 2024-04-20 17:05:50,261] Trial 1 finished with value: 0.7545846927159654 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.7563214317153626.
Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7366071428571429
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.6525821596244131
[I 2024-04-20 17:05:50,882] Trial 2 finished with value: 0.7545846927159654 and parameters: {'l1_ratio': 0.22692876841884668}. Be

Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6525821596244131
[I 2024-04-20 17:05:58,575] Trial 24 finished with value: 0.7563214317153626 and parameters: {'l1_ratio': 0.8064824763528503}. Best is trial 0 with value: 0.7563214317153626.
Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7366071428571429
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.6525821596244131
[I 2024-04-20 17:05:59,058] Trial 25 finished with value: 0.7545846927159654 and parameters: {'l1_ratio': 0.6339676394936953}. Best is trial 0 with value: 0.7563214317153626.
Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.6383928571428571
Fold 3 C-index: 0.7941176470588235
Fold 4 C-index: 0.8016877637130801
Fold 5 C-index: 0.647887323943662
[I 2024-04-20 17:05:59,205] Trial 26 finished with value: 0.7227374646920308 and parameters: {'l1_ratio': 0.015423757551295547}. Best is trial 0 with value: 0.7563214317153626.
Fold 1 C-index: 0.735

Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6525821596244131
[I 2024-04-20 17:06:06,135] Trial 48 finished with value: 0.7563214317153626 and parameters: {'l1_ratio': 0.8526883818058227}. Best is trial 0 with value: 0.7563214317153626.
Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6525821596244131
[I 2024-04-20 17:06:06,493] Trial 49 finished with value: 0.7563214317153626 and parameters: {'l1_ratio': 0.730055009401814}. Best is trial 0 with value: 0.7563214317153626.
Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7366071428571429
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.6525821596244131
[I 2024-04-20 17:06:06,920] Trial 50 finished with value: 0.7545846927159654 and parameters: {'l1_ratio': 0.47241766561900594}. Best is trial 0 with value: 0.7563214317153626.
Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.7366

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.18223089538135095
Fold 2 IBS: 0.17953058635719008
Fold 3 IBS: 0.15710637636572208
Fold 4 IBS: 0.14705004078562361
Fold 5 IBS: 0.2725427772516032
[I 2024-04-20 17:06:23,355] Trial 0 finished with value: 0.187692135228298 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.187692135228298.
Fold 1 IBS: 0.1821862002988621
Fold 2 IBS: 0.1796266625284713
Fold 3 IBS: 0.15734267606089006
Fold 4 IBS: 0.14728145985772745
Fold 5 IBS: 0.2725925913053688
[I 2024-04-20 17:06:23,732] Trial 1 finished with value: 0.18780591801026394 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.187692135228298.
Fold 1 IBS: 0.18218283176202407
Fold 2 IBS: 0.1796412294542746
Fold 3 IBS: 0.1573625510861116
Fold 4 IBS: 0.1473020530956881
Fold 5 IBS: 0.2725878481090234
[I 2024-04-20 17:06:24,293] Trial 2 finished with value: 0.18781530270142435 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.187692135228298.
Fold 1

Fold 3 IBS: 0.156872222110765
Fold 4 IBS: 0.14685332071526855
Fold 5 IBS: 0.2725237858784887
[I 2024-04-20 17:06:32,703] Trial 25 finished with value: 0.1875805124074397 and parameters: {'l1_ratio': 0.9099439614290462}. Best is trial 17 with value: 0.18751239753557997.
Fold 1 IBS: 0.1822247523708942
Fold 2 IBS: 0.1795662094311297
Fold 3 IBS: 0.15716070874522764
Fold 4 IBS: 0.14710382843507397
Fold 5 IBS: 0.27252429124360517
[I 2024-04-20 17:06:33,100] Trial 26 finished with value: 0.1877159580451861 and parameters: {'l1_ratio': 0.6258200052759627}. Best is trial 17 with value: 0.18751239753557997.
Fold 1 IBS: 0.18224877276585452
Fold 2 IBS: 0.17939690902708083
Fold 3 IBS: 0.1568766125262203
Fold 4 IBS: 0.1468179562409922
Fold 5 IBS: 0.272540234203548
[I 2024-04-20 17:06:33,541] Trial 27 finished with value: 0.18757609695273916 and parameters: {'l1_ratio': 0.9144784282298041}. Best is trial 17 with value: 0.18751239753557997.
Fold 1 IBS: 0.1822391690282693
Fold 2 IBS: 0.1794631280953686

Fold 3 IBS: 0.15691982750840544
Fold 4 IBS: 0.14686829028864845
Fold 5 IBS: 0.2725019968224739
[I 2024-04-20 17:06:42,507] Trial 50 finished with value: 0.18759426840847618 and parameters: {'l1_ratio': 0.8743196954547819}. Best is trial 17 with value: 0.18751239753557997.
Fold 1 IBS: 0.18225441586500854
Fold 2 IBS: 0.17933590719210243
Fold 3 IBS: 0.1568437243098027
Fold 4 IBS: 0.14675535719041946
Fold 5 IBS: 0.2725494403512185
[I 2024-04-20 17:06:42,841] Trial 51 finished with value: 0.1875477689817103 and parameters: {'l1_ratio': 0.9711488252676377}. Best is trial 17 with value: 0.18751239753557997.
Fold 1 IBS: 0.18225478681638807
Fold 2 IBS: 0.17934461863558518
Fold 3 IBS: 0.15684829781803014
Fold 4 IBS: 0.14675322963475146
Fold 5 IBS: 0.2725676193077559
[I 2024-04-20 17:06:43,152] Trial 52 finished with value: 0.18755371044250216 and parameters: {'l1_ratio': 0.9759519327420665}. Best is trial 17 with value: 0.18751239753557997.
Fold 1 IBS: 0.18225616070202216
Fold 2 IBS: 0.179317715

Fold 3 IBS: 0.15694321789100807
Fold 4 IBS: 0.1468576641263616
Fold 5 IBS: 0.2724844407720259
[I 2024-04-20 17:06:51,348] Trial 75 finished with value: 0.18759022414050994 and parameters: {'l1_ratio': 0.8991728554833402}. Best is trial 17 with value: 0.18751239753557997.
Fold 1 IBS: 0.18216275399195234
Fold 2 IBS: 0.1802289488354696
Fold 3 IBS: 0.1573965303596285
Fold 4 IBS: 0.14733515860809535
Fold 5 IBS: 0.27227111284228916
[I 2024-04-20 17:06:51,871] Trial 76 finished with value: 0.187878900927487 and parameters: {'l1_ratio': 0.11711434342988569}. Best is trial 17 with value: 0.18751239753557997.
Fold 1 IBS: 0.18223961760864696
Fold 2 IBS: 0.17945450096558654
Fold 3 IBS: 0.157030153389453
Fold 4 IBS: 0.14694109335553382
Fold 5 IBS: 0.27255757384157153
[I 2024-04-20 17:06:52,237] Trial 77 finished with value: 0.18764458783215837 and parameters: {'l1_ratio': 0.8295696347989644}. Best is trial 17 with value: 0.18751239753557997.
Fold 1 IBS: 0.18225459643154102
Fold 2 IBS: 0.17933287216

In [52]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [53]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.756
train_ibs:  0.188


#### Test

In [54]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [55]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
gbsg_times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(gbsg_times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, gbsg_times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.6964995386793018)

test_cindex : 0.582


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.9853615392538034)

test_ibs:  0.287


In [56]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [57]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                gbsg_times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(gbsg_times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, gbsg_times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-20 17:06:59,839] A new study created in memory with name: no-name-3e1b683d-6879-41eb-9feb-f5e82ae66a49


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.6339285714285714
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8544303797468354
Fold 5 C-index: 0.6408450704225352
[I 2024-04-20 17:07:02,451] Trial 0 finished with value: 0.7372744686944293 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7372744686944293.
Fold 1 C-index: 0.7012987012987013
Fold 2 C-index: 0.6875
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8607594936708861
Fold 5 C-index: 0.647887323943662
[I 2024-04-20 17:07:04,114] Trial 1 finished with value: 0.74027341750814 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'max_features': '

Fold 1 C-index: 0.70995670995671
Fold 2 C-index: 0.8392857142857143
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8755274261603375
Fold 5 C-index: 0.7230046948356808
[I 2024-04-20 17:07:22,687] Trial 16 finished with value: 0.7913196149300414 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 15, 'min_samples_leaf': 16, 'max_depth': 2, 'n_estimators': 62, 'oob_score': True, 'max_samples': 0.9622193970781673, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.07748782870655765, 'warm_start': True}. Best is trial 14 with value: 0.8012393138587386.
Fold 1 C-index: 0.6796536796536796
Fold 2 C-index: 0.8571428571428571
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.8839662447257384
Fold 5 C-index: 0.7417840375586855
[I 2024-04-20 17:07:23,285] Trial 17 finished with value: 0.7952544618554078 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 16, 'min_samples_leaf': 15, 'max_depth': 7, 'n_estimators': 127, 'oob_score': True, 'max_samples': 0.8172323984404596, 

Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.8571428571428571
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.9071729957805907
Fold 5 C-index: 0.7981220657276995
[I 2024-04-20 17:07:37,517] Trial 31 finished with value: 0.8102976168343801 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 7, 'min_samples_leaf': 7, 'max_depth': 12, 'n_estimators': 317, 'oob_score': True, 'max_samples': 0.8965443067001324, 'max_features': None, 'min_weight_fraction_leaf': 0.04869831874272522, 'warm_start': True}. Best is trial 30 with value: 0.8141045941707695.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.8571428571428571
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.9029535864978903
Fold 5 C-index: 0.7746478873239436
[I 2024-04-20 17:07:38,723] Trial 32 finished with value: 0.8044151254239035 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 6, 'min_samples_leaf': 7, 'max_depth': 13, 'n_estimators': 317, 'oob_score': True, 'max_samples': 0.7822231835771817,

...

Fold 1 IBS: 0.21398563901141657
Fold 2 IBS: 0.22134596740550985
Fold 3 IBS: 0.2048025990579434
Fold 4 IBS: 0.22461415055237854
Fold 5 IBS: 0.21851803782809134
[I 2024-04-20 17:09:53,536] Trial 32 finished with value: 0.21665327877106794 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 10, 'min_samples_leaf': 3, 'max_depth': 18, 'n_estimators': 451, 'oob_score': False, 'max_samples': 0.7396444271446992, 'max_features': None, 'min_weight_fraction_leaf': 0.4501495537267681}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.19337121417130673
Fold 2 IBS: 0.19511882700454072
Fold 3 IBS: 0.18642967451701625
Fold 4 IBS: 0.1895345717679471
Fold 5 IBS: 0.21856358775020004
[I 2024-04-20 17:09:56,380] Trial 33 finished with value: 0.19660357504220216 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 12, 'min_samples_leaf': 1, 'max_depth': 19, 'n_estimators': 493, 'oob_score': False, 'max_samples': 0.9933327757837453, 'max_features': None, 'min_weight_fraction_leaf

Fold 1 IBS: 0.21398162469590376
Fold 2 IBS: 0.22136781149843104
Fold 3 IBS: 0.20480753036595514
Fold 4 IBS: 0.22461586717115936
Fold 5 IBS: 0.21858421480458282
[I 2024-04-20 17:10:32,489] Trial 48 finished with value: 0.21667140970720644 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 459, 'oob_score': False, 'max_samples': 0.8399357435042919, 'max_features': None, 'min_weight_fraction_leaf': 0.43949783817145754}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.18607291644289184
Fold 2 IBS: 0.1997166990883884
Fold 3 IBS: 0.17871196581310303
Fold 4 IBS: 0.1866755623669813
Fold 5 IBS: 0.21401906861191788
[I 2024-04-20 17:10:35,015] Trial 49 finished with value: 0.19303924246465648 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 6, 'min_samples_leaf': 20, 'max_depth': 6, 'n_estimators': 480, 'oob_score': False, 'max_samples': 0.9976678104279734, 'max_features': 'auto', 'min_weight_fraction_lea

Fold 1 IBS: 0.17214163753608527
Fold 2 IBS: 0.19020363580055616
Fold 3 IBS: 0.16550959046326688
Fold 4 IBS: 0.16526794959392554
Fold 5 IBS: 0.22833087784482511
[I 2024-04-20 17:11:06,242] Trial 64 finished with value: 0.1842907382477318 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 14, 'min_samples_leaf': 7, 'max_depth': 18, 'n_estimators': 359, 'oob_score': False, 'max_samples': 0.8662053800926933, 'max_features': None, 'min_weight_fraction_leaf': 0.39542258978805284}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.2139792007492792
Fold 2 IBS: 0.22131668306329724
Fold 3 IBS: 0.20481655835559517
Fold 4 IBS: 0.22465389902940344
Fold 5 IBS: 0.2186107338002615
[I 2024-04-20 17:11:08,128] Trial 65 finished with value: 0.21667541499956733 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 12, 'min_samples_leaf': 8, 'max_depth': 20, 'n_estimators': 370, 'oob_score': False, 'max_samples': 0.8126605473414451, 'max_features': None, 'min_weight_fraction_leaf'

Fold 1 IBS: 0.21398666839515476
Fold 2 IBS: 0.22133561292920703
Fold 3 IBS: 0.20476084552868884
Fold 4 IBS: 0.22460858228328426
Fold 5 IBS: 0.21850495327987215
[I 2024-04-20 17:11:38,298] Trial 80 finished with value: 0.2166393324832414 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 14, 'min_samples_leaf': 6, 'max_depth': 16, 'n_estimators': 442, 'oob_score': True, 'max_samples': 0.7543724309332009, 'max_features': None, 'min_weight_fraction_leaf': 0.43164085935118673}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.1718240887458899
Fold 2 IBS: 0.191437443203446
Fold 3 IBS: 0.16491345446124533
Fold 4 IBS: 0.16398408770898562
Fold 5 IBS: 0.22656733476064983
[I 2024-04-20 17:11:40,253] Trial 81 finished with value: 0.18374528177604332 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 14, 'min_samples_leaf': 8, 'max_depth': 18, 'n_estimators': 355, 'oob_score': False, 'max_samples': 0.8723363913158229, 'max_features': None, 'min_weight_fraction_leaf': 

Fold 1 IBS: 0.18206223269633873
Fold 2 IBS: 0.18820585180072097
Fold 3 IBS: 0.17369714014143833
Fold 4 IBS: 0.1751220212365198
Fold 5 IBS: 0.22437209861320465
[I 2024-04-20 17:12:14,882] Trial 96 finished with value: 0.1886918688976445 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 8, 'n_estimators': 448, 'oob_score': False, 'max_samples': 0.8430407084230132, 'max_features': None, 'min_weight_fraction_leaf': 0.41419337591128474}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.17261810934600746
Fold 2 IBS: 0.19291443628076307
Fold 3 IBS: 0.16469348295311864
Fold 4 IBS: 0.16293280131529528
Fold 5 IBS: 0.2265701418039605
[I 2024-04-20 17:12:17,774] Trial 97 finished with value: 0.183945794339829 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 15, 'min_samples_leaf': 3, 'max_depth': 9, 'n_estimators': 499, 'oob_score': False, 'max_samples': 0.8784577990568653, 'max_features': None, 'min_weight_fraction_leaf': 0

In [58]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [59]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.826
train_ibs:  0.183


#### Test

In [60]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

In [61]:
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])
 
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
gbsg_times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(gbsg_times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, gbsg_times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=19, max_features='log2', max_leaf_nodes=15,
                     max_samples=0.9995515317027949, min_samples_split=18,
                     min_weight_fraction_leaf=0.032175807966512454,
                     n_estimators=115, oob_score=True, random_state=123,
                     warm_start=True)

test_cindex:  0.653


RandomSurvivalForest(max_depth=20, max_features=None, max_leaf_nodes=13,
                     max_samples=0.9248759152889626, min_samples_split=2,
                     min_weight_fraction_leaf=0.41894325026727597,
                     n_estimators=396, random_state=123)

test_ibs:  0.206


In [62]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [63]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [64]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                gbsg_times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(gbsg_times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, gbsg_times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-20 17:12:23,456] A new study created in memory with name: no-name-2c34a0de-f269-46b8-8aa4-6130fd75326a


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.8080357142857143
Fold 3 C-index: 0.8480392156862745
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6948356807511737
[I 2024-04-20 17:12:24,070] Trial 0 finished with value: 0.7836129950691764 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7836129950691764.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-20 17:12:25,328] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.8013392857142857
Fold 3 C-index: 0.8333333333333334
Fold 4 C-index: 0.8248945147679325
Fold 5 C-index: 0.6267605633802817
[I 2024-04-20 17:12:40,052] Trial 16 finished with value: 0.77051229268592 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is trial 12 with value: 0.7914452579440178.
Fold 1 C-index: 0.7575757575757576
Fold 2 C-index: 0.7901785714285714
Fold 3 C-index: 0.8333333333333334
Fold 4 C-index: 0.8185654008438819
Fold 5 C-index: 0.6619718309859155
[I 2024-04-20 17:12:40,535] Trial 17 finished with value: 0.7723249788334919 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 12, 'n_estimators': 318, 'oob_score': False, 'warm_start': True, 'max_featur

Fold 1 C-index: 0.7229437229437229
Fold 2 C-index: 0.8169642857142857
Fold 3 C-index: 0.8529411764705882
Fold 4 C-index: 0.8248945147679325
Fold 5 C-index: 0.6384976525821596
[I 2024-04-20 17:12:51,898] Trial 31 finished with value: 0.7712482704957379 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 5, 'min_samples_leaf': 16, 'max_depth': 13, 'n_estimators': 500, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.9319495980432307, 'min_weight_fraction_leaf': 0.028814153765248245}. Best is trial 12 with value: 0.7914452579440178.
Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.8482142857142857
Fold 3 C-index: 0.8480392156862745
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6619718309859155
[I 2024-04-20 17:12:53,217] Trial 32 finished with value: 0.7875856659621985 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 7, 'min_samples_leaf': 9, 'max_depth': 13, 'n_estimators': 459, 'oob_score': True, 'warm_start': True, 'max_features'

...

Fold 1 IBS: 0.16792251267623906
Fold 2 IBS: 0.2174484716230587
Fold 3 IBS: 0.16273781989489802
Fold 4 IBS: 0.1558940391227505
Fold 5 IBS: 0.23222422254692077
[I 2024-04-20 17:14:59,201] Trial 30 finished with value: 0.18724541317277338 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 6, 'min_samples_leaf': 2, 'max_depth': 12, 'n_estimators': 455, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.7293028765390641, 'min_weight_fraction_leaf': 0.08037045697723012}. Best is trial 23 with value: 0.1855727116611247.
Fold 1 IBS: 0.16646920043412164
Fold 2 IBS: 0.22144555868438975
Fold 3 IBS: 0.1612585234507463
Fold 4 IBS: 0.15276461377317607
Fold 5 IBS: 0.23760424924495455
[I 2024-04-20 17:15:00,396] Trial 31 finished with value: 0.18790842911747765 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 5, 'n_estimators': 209, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.83

Fold 1 IBS: 0.16512803721910052
Fold 2 IBS: 0.21893230349393553
Fold 3 IBS: 0.1593836385256823
Fold 4 IBS: 0.15058205502042205
Fold 5 IBS: 0.23336241815587846
[I 2024-04-20 17:15:19,737] Trial 45 finished with value: 0.18547769048300378 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 6, 'min_samples_leaf': 3, 'max_depth': 12, 'n_estimators': 276, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6931500413934508, 'min_weight_fraction_leaf': 0.021457257714753555}. Best is trial 32 with value: 0.18528124104479504.
Fold 1 IBS: 0.1659546836323674
Fold 2 IBS: 0.2153197227930472
Fold 3 IBS: 0.16360867863772716
Fold 4 IBS: 0.15557836846786757
Fold 5 IBS: 0.233640633124344
[I 2024-04-20 17:15:21,253] Trial 46 finished with value: 0.18682041733107066 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 13, 'min_samples_leaf': 6, 'max_depth': 13, 'n_estimators': 283, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0

Fold 1 IBS: 0.17342492293804432
Fold 2 IBS: 0.2110732537830341
Fold 3 IBS: 0.16831819527481992
Fold 4 IBS: 0.16873369129336893
Fold 5 IBS: 0.22655614890084055
[I 2024-04-20 17:15:48,957] Trial 60 finished with value: 0.18962124243802156 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 4, 'min_samples_leaf': 5, 'max_depth': 14, 'n_estimators': 439, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.5735294094022945, 'min_weight_fraction_leaf': 0.10077092974890593}. Best is trial 53 with value: 0.18257594357755064.
Fold 1 IBS: 0.16275315104094723
Fold 2 IBS: 0.22028966274589679
Fold 3 IBS: 0.16253743592400494
Fold 4 IBS: 0.15352895984062664
Fold 5 IBS: 0.23021227496706434
[I 2024-04-20 17:15:51,149] Trial 61 finished with value: 0.18586429690370798 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 4, 'min_samples_leaf': 3, 'max_depth': 11, 'n_estimators': 412, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples':

Fold 1 IBS: 0.16278536069617075
Fold 2 IBS: 0.22061911570320727
Fold 3 IBS: 0.15948095174532076
Fold 4 IBS: 0.1500453215492157
Fold 5 IBS: 0.23108140998200039
[I 2024-04-20 17:16:24,028] Trial 75 finished with value: 0.18480243193518295 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 13, 'n_estimators': 463, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.6487943567829829, 'min_weight_fraction_leaf': 0.011079149496446542}. Best is trial 53 with value: 0.18257594357755064.
Fold 1 IBS: 0.1649931201311961
Fold 2 IBS: 0.2191098546301487
Fold 3 IBS: 0.1604180424655896
Fold 4 IBS: 0.15133070246635846
Fold 5 IBS: 0.23270813963892614
[I 2024-04-20 17:16:27,345] Trial 76 finished with value: 0.18571197186644378 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 13, 'n_estimators': 496, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0

Fold 1 IBS: 0.18162653052097805
Fold 2 IBS: 0.20612651784753333
Fold 3 IBS: 0.1771784733434652
Fold 4 IBS: 0.18941755917999278
Fold 5 IBS: 0.21614311489575638
[I 2024-04-20 17:16:58,173] Trial 90 finished with value: 0.19409843915754516 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 6, 'min_samples_leaf': 1, 'max_depth': 20, 'n_estimators': 362, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.7635642567316503, 'min_weight_fraction_leaf': 0.280081595660318}. Best is trial 53 with value: 0.18257594357755064.
Fold 1 IBS: 0.16186089564069253
Fold 2 IBS: 0.21899259491528042
Fold 3 IBS: 0.15921485444738478
Fold 4 IBS: 0.148193042465612
Fold 5 IBS: 0.23190417972474658
[I 2024-04-20 17:17:00,373] Trial 91 finished with value: 0.18403311343874326 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 5, 'min_samples_leaf': 1, 'max_depth': 19, 'n_estimators': 323, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.789

In [65]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [66]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.794
train_ibs:  0.183


#### Test

In [67]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [68]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
gbsg_times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(gbsg_times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, gbsg_times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=12, max_features=None, max_leaf_nodes=6,
                   max_samples=0.7030590704740037, min_samples_leaf=6,
                   min_samples_split=13,
                   min_weight_fraction_leaf=0.020016091327628317,
                   n_estimators=273, random_state=123, warm_start=True)

C-index score: 0.618


ExtraSurvivalTrees(max_depth=13, max_features='log2', max_leaf_nodes=6,
                   max_samples=0.7048480399273186, min_samples_leaf=1,
                   min_samples_split=12,
                   min_weight_fraction_leaf=0.0010291888027469595,
                   n_estimators=485, random_state=123, warm_start=True)

IBS: 0.215


In [69]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [70]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                gbsg_times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(gbsg_times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, gbsg_times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-20 17:17:19,313] A new study created in memory with name: no-name-3229b0f1-758b-41e8-8f15-2a340747ffd7


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-20 17:17:29,612] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-20 17:17:35,340] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-20 17:20:32,669] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7334874350516556.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-20 17:20:56,206] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-20 17:25:01,049] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 9 with value: 0.7334874350516556.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.6004464285714286
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.6103286384976526
[I 2024-04-20 17:25:28,053] Trial 26 finished with value: 0.5421550134138162 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446,

...

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-20 17:55:05,154] Trial 55 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9027683411994925, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.13472529918097312, 'n_estimators': 381, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.36938177041618503, 'max_features': 'auto', 'min_impurity_decrease': 1.1016843774656315e-07, 'validation_fraction': 0.898549324711475, 'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 53 with value: 0.2150432178150902.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-20 17:55:28,721] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.8324789538052517, 'learning_rate': 0.0985092209

Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609563
[I 2024-04-20 18:00:11,712] Trial 66 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9104313304151431, 'learning_rate': 0.014815622348446418, 'dropout_rate': 0.1415896279444537, 'n_estimators': 149, 'criterion': 'squared_error', 'ccp_alpha': 0.7060910424624821, 'min_weight_fraction_leaf': 0.2796836959018326, 'max_features': 'auto', 'min_impurity_decrease': 5.127880425748818e-06, 'validation_fraction': 0.916501041728546, 'min_samples_split': 19, 'max_leaf_nodes': 16, 'min_samples_leaf': 18, 'max_depth': 5}. Best is trial 53 with value: 0.2150432178150902.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-20 18:00:38,543] Trial 67 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.802369464636257, 'learning_rate': 0.0041755238500

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609566
[I 2024-04-20 18:06:05,759] Trial 77 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8921532985730489, 'learning_rate': 0.021931279910218017, 'dropout_rate': 0.19284931428290142, 'n_estimators': 490, 'criterion': 'squared_error', 'ccp_alpha': 0.5905545096948588, 'min_weight_fraction_leaf': 0.07057220045251793, 'max_features': 'log2', 'min_impurity_decrease': 1.637559261311236e-05, 'validation_fraction': 0.8150762480319586, 'min_samples_split': 5, 'max_leaf_nodes': 13, 'min_samples_leaf': 8, 'max_depth': 5}. Best is trial 53 with value: 0.2150432178150902.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609563
[I 2024-04-20 18:06:36,999] Trial 78 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.7443996478445325, 'learning_rate': 0.086932868663

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609566
[I 2024-04-20 18:11:25,183] Trial 88 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9447987915285996, 'learning_rate': 0.014159070275535094, 'dropout_rate': 0.18102387516231752, 'n_estimators': 470, 'criterion': 'squared_error', 'ccp_alpha': 0.5935000311483954, 'min_weight_fraction_leaf': 0.2151508441811604, 'max_features': 'auto', 'min_impurity_decrease': 3.72786238165322e-07, 'validation_fraction': 0.898328514849679, 'min_samples_split': 18, 'max_leaf_nodes': 18, 'min_samples_leaf': 12, 'max_depth': 2}. Best is trial 53 with value: 0.2150432178150902.
Fold 1 IBS: 0.21292314886383587
Fold 2 IBS: 0.2213935582997429
Fold 3 IBS: 0.20349492943164285
Fold 4 IBS: 0.2234443788901035
Fold 5 IBS: 0.2178743294294726
[I 2024-04-20 18:11:49,602] Trial 89 finished with value: 0.21582606898295956 and parameters: {'subsample': 0.9777744983689288, 'learning_rate': 0.010383571834622

Fold 3 IBS: 0.20237857642828805
Fold 4 IBS: 0.22105646622231043
Fold 5 IBS: 0.21749535106188492
[I 2024-04-20 18:15:48,266] Trial 99 finished with value: 0.21481876339222578 and parameters: {'subsample': 0.8569271041566342, 'learning_rate': 0.01377474153403631, 'dropout_rate': 0.2044520049398774, 'n_estimators': 378, 'criterion': 'squared_error', 'ccp_alpha': 0.004422201299274769, 'min_weight_fraction_leaf': 0.13940114558509004, 'max_features': 'auto', 'min_impurity_decrease': 0.00015837847703956763, 'validation_fraction': 0.9580200141092102, 'min_samples_split': 13, 'max_leaf_nodes': 17, 'min_samples_leaf': 13, 'max_depth': 2}. Best is trial 99 with value: 0.21481876339222578.


* Best trial for IBS: 
 FrozenTrial(number=99, state=TrialState.COMPLETE, values=[0.21481876339222578], datetime_start=datetime.datetime(2024, 4, 20, 18, 15, 25, 784864), datetime_complete=datetime.datetime(2024, 4, 20, 18, 15, 48, 265093), params={'subsample': 0.8569271041566342, 'learning_rate': 0.0137747415

In [71]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [72]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.745
train_ibs:  0.215


#### Test

In [73]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [74]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
gbsg_times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(gbsg_times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, gbsg_times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.03063070291051248,
                                 criterion='squared_error',
                                 dropout_rate=0.2576884847115747,
                                 learning_rate=0.007359366951045268,
                                 max_depth=16, max_features=1,
                                 max_leaf_nodes=16,
                                 min_impurity_decrease=1.3903906488794697e-07,
                                 min_samples_leaf=14, min_samples_split=20,
                                 min_weight_fraction_leaf=0.38180626899357545,
                                 n_estimators=96, random_state=123,
                                 subsample=0.8938290428827321,
                                 validation_fraction=0.9577535215137098)

C-index score: 0.603


GradientBoostingSurvivalAnalysis(ccp_alpha=0.004422201299274769,
                                 criterion='squared_error',
                                 dropout_rate=0.2044520049398774,
                                 learning_rate=0.01377474153403631, max_depth=2,
                                 max_features='auto', max_leaf_nodes=17,
                                 min_impurity_decrease=0.00015837847703956763,
                                 min_samples_leaf=13, min_samples_split=13,
                                 min_weight_fraction_leaf=0.13940114558509004,
                                 n_estimators=378, random_state=123,
                                 subsample=0.8569271041566342,
                                 validation_fraction=0.9580200141092102)

IBS: 0.22


In [75]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [76]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [77]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                gbsg_times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(gbsg_times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, gbsg_times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-20 18:15:53,338] A new study created in memory with name: no-name-aa3d9c0c-a182-40ed-be54-7526fc2bdee6


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.609375
Fold 3 C-index: 0.7450980392156863
Fold 4 C-index: 0.7848101265822784
Fold 5 C-index: 0.6103286384976526
[I 2024-04-20 18:15:53,932] Trial 0 finished with value: 0.6849872959240584 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6849872959240584.
Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.609375
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.7848101265822784
Fold 5 C-index: 0.6103286384976526
[I 2024-04-20 18:15:58,493] Trial 1 finished with value: 0.6943010214142545 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 1 with value: 0.6943010214142545.
Fold 1 C-index: 0.7251082251082251
Fold 2 C-index: 0.609375
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.7848101265822784


Fold 1 C-index: 0.7835497835497836
Fold 2 C-index: 0.6830357142857143
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6173708920187794
[I 2024-04-20 18:16:40,976] Trial 19 finished with value: 0.7413693370425853 and parameters: {'subsample': 0.34314044045637715, 'dropout_rate': 0.1962189134479061, 'n_estimators': 426, 'learning_rate': 0.09857240282031608}. Best is trial 14 with value: 0.7520577769511994.
Fold 1 C-index: 0.7835497835497836
Fold 2 C-index: 0.6919642857142857
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6244131455399061
[I 2024-04-20 18:16:41,560] Trial 20 finished with value: 0.744563502032525 and parameters: {'subsample': 0.2628688080573812, 'dropout_rate': 0.43503165140528216, 'n_estimators': 112, 'learning_rate': 0.0794193638388259}. Best is trial 14 with value: 0.7520577769511994.
Fold 1 C-index: 0.7575757575757576
Fold 2 C-index: 0.7008928571428571
Fold 3 C-index: 0.803921568627451
Fo

Fold 3 C-index: 0.8063725490196079
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6126760563380281
[I 2024-04-20 18:17:18,701] Trial 38 finished with value: 0.7442644035198804 and parameters: {'subsample': 0.16295147397293233, 'dropout_rate': 0.5110436492508221, 'n_estimators': 82, 'learning_rate': 0.07364542219225001}. Best is trial 34 with value: 0.7541296003076832.
Fold 1 C-index: 0.7835497835497836
Fold 2 C-index: 0.6919642857142857
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6126760563380281
[I 2024-04-20 18:17:19,634] Trial 39 finished with value: 0.7422160841921494 and parameters: {'subsample': 0.25410451878922247, 'dropout_rate': 0.14050020390445545, 'n_estimators': 159, 'learning_rate': 0.06709978541578802}. Best is trial 34 with value: 0.7541296003076832.
Fold 1 C-index: 0.7251082251082251
Fold 2 C-index: 0.6919642857142857
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.8227848101265823
Fold 5 C-index: 0.6126760563380281

...

Fold 4 IBS: 0.2017180216457497
Fold 5 IBS: 0.2127622739815413
[I 2024-04-20 18:20:24,057] Trial 19 finished with value: 0.19707996037517952 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9930703343219778, 'n_estimators': 51, 'learning_rate': 0.04096883532946153}. Best is trial 19 with value: 0.19707996037517952.
Fold 1 IBS: 0.18844165738979637
Fold 2 IBS: 0.20472965504337676
Fold 3 IBS: 0.1819421294378562
Fold 4 IBS: 0.20465119827707254
Fold 5 IBS: 0.21298081915188694
[I 2024-04-20 18:20:24,312] Trial 20 finished with value: 0.19854909185999775 and parameters: {'subsample': 0.1435040576337751, 'dropout_rate': 0.7933084651006226, 'n_estimators': 43, 'learning_rate': 0.04127909023805986}. Best is trial 19 with value: 0.19707996037517952.
Fold 1 IBS: 0.19558104368646592
Fold 2 IBS: 0.20382011576839348
Fold 3 IBS: 0.18712698518091442
Fold 4 IBS: 0.20944879242499856
Fold 5 IBS: 0.21303933620208118
[I 2024-04-20 18:20:24,522] Trial 21 finished with value: 0.201803254652

Fold 5 IBS: 0.2332838201025736
[I 2024-04-20 18:20:36,654] Trial 38 finished with value: 0.20744641732991104 and parameters: {'subsample': 0.1710779175611994, 'dropout_rate': 0.9288534916051375, 'n_estimators': 197, 'learning_rate': 0.036023800474466655}. Best is trial 30 with value: 0.19594477372590474.
Fold 1 IBS: 0.1876140504745091
Fold 2 IBS: 0.20563381823568802
Fold 3 IBS: 0.17202903977012926
Fold 4 IBS: 0.2067894022485534
Fold 5 IBS: 0.21264550977192898
[I 2024-04-20 18:20:36,948] Trial 39 finished with value: 0.19694236410016178 and parameters: {'subsample': 0.3537909584345821, 'dropout_rate': 0.8744094300062906, 'n_estimators': 59, 'learning_rate': 0.04711870419619728}. Best is trial 30 with value: 0.19594477372590474.
Fold 1 IBS: 0.18448495606229867
Fold 2 IBS: 0.2155619099375562
Fold 3 IBS: 0.16824012001923247
Fold 4 IBS: 0.21346674211873537
Fold 5 IBS: 0.21402742087944407
[I 2024-04-20 18:20:37,223] Trial 40 finished with value: 0.19915622980345335 and parameters: {'subsampl

Fold 5 IBS: 0.2667682961494491
[I 2024-04-20 18:20:44,949] Trial 57 finished with value: 0.246586139989994 and parameters: {'subsample': 0.32826887806420246, 'dropout_rate': 0.9933561701936838, 'n_estimators': 283, 'learning_rate': 0.046115905841529976}. Best is trial 30 with value: 0.19594477372590474.
Fold 1 IBS: 0.20860918635569756
Fold 2 IBS: 0.21590342208816954
Fold 3 IBS: 0.19882549664838997
Fold 4 IBS: 0.22138259267967494
Fold 5 IBS: 0.21576315984193745
[I 2024-04-20 18:20:45,195] Trial 58 finished with value: 0.2120967715227739 and parameters: {'subsample': 0.5286310530837229, 'dropout_rate': 0.922313337459929, 'n_estimators': 41, 'learning_rate': 0.009368892992001188}. Best is trial 30 with value: 0.19594477372590474.
Fold 1 IBS: 0.16721506156919788
Fold 2 IBS: 0.20811485477202274
Fold 3 IBS: 0.1673944010677237
Fold 4 IBS: 0.19470174832450804
Fold 5 IBS: 0.2186581694007046
[I 2024-04-20 18:20:45,677] Trial 59 finished with value: 0.1912168470268314 and parameters: {'subsample'

Fold 5 IBS: 0.21640786738462428
[I 2024-04-20 18:20:59,986] Trial 76 finished with value: 0.1904383507221824 and parameters: {'subsample': 0.12400746444313113, 'dropout_rate': 0.3383709467175438, 'n_estimators': 215, 'learning_rate': 0.017149355016391896}. Best is trial 76 with value: 0.1904383507221824.
Fold 1 IBS: 0.1724380363374518
Fold 2 IBS: 0.20474762692041973
Fold 3 IBS: 0.1715063350229059
Fold 4 IBS: 0.19856558172691882
Fold 5 IBS: 0.2145540355925452
[I 2024-04-20 18:21:01,344] Trial 77 finished with value: 0.19236232312004828 and parameters: {'subsample': 0.17633990236538394, 'dropout_rate': 0.3283434771119012, 'n_estimators': 211, 'learning_rate': 0.015746054804451973}. Best is trial 76 with value: 0.1904383507221824.
Fold 1 IBS: 0.16981201583729438
Fold 2 IBS: 0.2017243145055686
Fold 3 IBS: 0.17165846403202692
Fold 4 IBS: 0.1944387983912983
Fold 5 IBS: 0.21589811641276688
[I 2024-04-20 18:21:02,720] Trial 78 finished with value: 0.19070634183579102 and parameters: {'subsampl

Fold 4 IBS: 0.21188491694556194
Fold 5 IBS: 0.21410930497209688
[I 2024-04-20 18:21:29,498] Trial 95 finished with value: 0.20389809513072327 and parameters: {'subsample': 0.13325875598093934, 'dropout_rate': 0.14328453982452782, 'n_estimators': 227, 'learning_rate': 0.0051619495422841905}. Best is trial 82 with value: 0.18932597122094488.
Fold 1 IBS: 0.18595695995906586
Fold 2 IBS: 0.2026086142619695
Fold 3 IBS: 0.17977537744545175
Fold 4 IBS: 0.20420808257142417
Fold 5 IBS: 0.2125609356574605
[I 2024-04-20 18:21:30,778] Trial 96 finished with value: 0.19702199397907433 and parameters: {'subsample': 0.1576853345960427, 'dropout_rate': 0.24198802409859813, 'n_estimators': 193, 'learning_rate': 0.01023216088648701}. Best is trial 82 with value: 0.18932597122094488.
Fold 1 IBS: 0.18656302725281798
Fold 2 IBS: 0.20311422177846558
Fold 3 IBS: 0.18168069964582573
Fold 4 IBS: 0.2048176795886752
Fold 5 IBS: 0.213526515687183
[I 2024-04-20 18:21:32,763] Trial 97 finished with value: 0.19794042

In [78]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [79]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.754
train_ibs:  0.189


#### Test

In [80]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [81]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
gbsg_times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(gbsg_times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, gbsg_times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.165569093503576,
                                              learning_rate=0.07576716957337241,
                                              n_estimators=142,
                                              random_state=123,
                                              subsample=0.10904393649144295)

C-index score: 0.56


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.20981564910049627,
                                              learning_rate=0.017066518914446174,
                                              n_estimators=226,
                                              random_state=123,
                                              subsample=0.11263052409999037)

IBS: 0.205


In [82]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [83]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.826,1.0
ExtraSurvivalTrees,0.794,2.0
CoxPH,0.758,3.0
CoxLasso,0.756,4.5
CoxElastic,0.756,4.5
ComponentwiseGradientBoosting,0.754,6.0
GradientBoosting,0.745,7.0
CoxRidge,0.718,8.0


In [84]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.183,1.5
ExtraSurvivalTrees,0.183,1.5
CoxLasso,0.188,3.5
CoxElastic,0.188,3.5
ComponentwiseGradientBoosting,0.189,5.0
GradientBoosting,0.215,6.0
CoxRidge,0.217,7.0
CoxPH,0.360,8.0


In [85]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
Randomsurvivalforest,0.653,1.0
CoxRidge,0.624,2.0
ExtraSurvivalTrees,0.618,3.0
GradientBoosting,0.603,4.0
CoxLasso,0.583,5.0
CoxElastic,0.582,6.0
ComponentwiseGradientBoosting,0.560,7.0


In [86]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
ComponentwiseGradientBoosting,0.205,1.0
Randomsurvivalforest,0.206,2.0
ExtraSurvivalTrees,0.215,3.0
GradientBoosting,0.220,4.0
CoxRidge,0.221,5.0
CoxLasso,0.287,6.5
CoxElastic,0.287,6.5


In [87]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/os/minmax/no_selection/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d1_os_minmax_no_selection_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [88]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-20
